# SIM V3 · Outdoor · Phase B + C + D — one run
**Directional** outdoor surrogate driven by **real base stations on real buildings** (`fw_bs_catalog`): building-mounted panel/sector sources, 10-channel input (the 9 material/Tx/freq channels + a directivity channel), held-out real station (**Forte Hall**) for g2. Same one-session, local-disk, CuPy+AMP flow as indoor. Pick a **GPU runtime**.

### 1 · Clone the repo + mount Drive

In [ ]:
# /content is wiped on every runtime restart, so (re)clone the code here.
import os, getpass
REPO_ROOT = '/content/indoor-walk-test'
if not os.path.isdir(os.path.join(REPO_ROOT, 'Physics Engine')):
    tok = getpass.getpass('GitHub token (repo read): ')
    os.system(f'git clone --depth 1 https://{tok}@github.com/cgm2179/indoor-walk-test.git "{REPO_ROOT}"')
try:
    from google.colab import drive; drive.mount('/content/drive')   # only for the final save
except ModuleNotFoundError:
    pass
print('repo present:', os.path.isdir(os.path.join(REPO_ROOT, 'Physics Engine', '2D', 'SIM V3')))

### 2 · Put SIM V3 on the path

In [ ]:
import sys, os
SIMV3 = os.path.join(REPO_ROOT, 'Physics Engine', '2D', 'SIM V3')
assert os.path.exists(os.path.join(SIMV3, '_bootstrap.py')), f'clone missing — re-run cell 1: {SIMV3}'
sys.path.insert(0, SIMV3); os.chdir(SIMV3)
import torch; print('SIM V3 =', SIMV3, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

### 3 · Build the georeferenced NoMa city grid (from the committed OBJ)
The city grid is gitignored (~360 MB), so build it here from `Data/models/NoMa_DC/NoMa_DC_buildings.obj` (~1–3 min, once per runtime). `load_georef_city` injects the FCC-HQ anchor in-memory, so no separate patch step.

In [ ]:
import subprocess, sys, _bootstrap as B
grid_npy = B.CITY_DIR / 'material_grid.npy'
if not grid_npy.exists():
    subprocess.run([sys.executable, str(B.SIM3D / 'voxelize_city.py'), '--no-preview'],
                   cwd=REPO_ROOT, check=True)                 # add --bbox-lonlat -77.0144 38.900 -77.0005 38.908 to crop/speed up
print('city grid ready:', grid_npy.exists(), '->', grid_npy)

### 4 · Parameters (cellular bands with real stations; Forte Hall held out)

In [ ]:
BANDS = ['TMO_B71_617', 'ATT/TMO/VZW_B12/B13/B14_746', 'VZW_B5/B26_885',
         'ATT/TMO/VZW_B2_1965', 'ATT/TMO/VZW_B4/B65/B66_2160', 'ATT_B30_2355',
         'TMO_n41_2508', 'VZW_n77/n78_3710', 'VZW_n77_3809', 'WLAN_WiFi_2442']
#   Network_Band(s)_Freq · every band a station picked up, grouped only within 30 MHz
#   (slashes = lumped). 9 cellular clusters (4G+5G) + artificial WLAN = one FDTD each.
HOLDOUT = None                  # None = train on ALL 6 stations (BS2-BS7)
VAL_SITE = 'BS7_forte_hall'     # station for the Phase-D g2 check (in-sample when HOLDOUT=None)
ANTENNA_KINDS = ['panel', 'dish', 'yagi', 'lpda', 'horn', 'slot', 'patch', 'loop', 'fractal']  # 2D antenna waveforms
OUT   = '/content/fw_data_bs'
CKPT  = '/content/fw_bs.pt'
EPOCHS, BASE, BATCH = 80, 32, 16
BOXES_PER_STATION = 16          # 24 (site,band) x 9 antennas x 16 boxes ~ 3.5k samples
MAX_CELLS = 6_000_000           # A100; on T4 use ~1_600_000
CITY_DIR = None                 # None = committed OBJ grid; REPO '.../SIM V1 3D/city/NoMa_DC_osm' for real OSM


### 4c · Use the REAL OSM city grid (aligned to true lon/lat)
The BlenderGIS OBJ model is simplified and can look mis-registered vs OSM. `voxelize_gpkg` places every building at its real lon/lat, so base stations land on the correct blocks. This builds it from `building.032010.gpkg` if present and points `CITY_DIR` at it; otherwise it falls back to the committed OBJ grid.

In [ ]:
import os, subprocess, sys, _bootstrap as B
USE_OSM = True                       # set False to force the committed OBJ grid
GPKG = next((p for p in ['/content/drive/MyDrive/building.032010.gpkg',
                         '/content/building.032010.gpkg',
                         os.path.expanduser('~/Downloads/building.032010.gpkg')]
             if os.path.exists(p)), None)
OSM_DIR = str(B.SIM3D / 'city' / 'NoMa_DC_osm')
if USE_OSM and GPKG:
    if not os.path.exists(os.path.join(OSM_DIR, 'material_grid.npy')):
        subprocess.run(['pip','-q','install','geopandas','pyogrio','shapely'])
        subprocess.run([sys.executable, str(B.SIM3D/'voxelize_gpkg.py'), '--gpkg', GPKG, '--no-preview'], check=True)
    CITY_DIR = OSM_DIR; print('CITY_DIR = real OSM grid ->', CITY_DIR)
else:
    CITY_DIR = None; print('OSM gpkg not found -> committed OBJ grid (upload building.032010.gpkg to /content or Drive, or set USE_OSM=False)')


# ---- Map + CRS of the grid the Dataset & U-Net see (all 6 stations · 4G/5G/WLAN · antennas) ----
import numpy as np, matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from collections import defaultdict
import _bootstrap as B, voxelize_city as VC, city_georef as CG
import fw_bs_catalog as fbs
from bands_v3 import get

grid, man, sts = fbs.load_stations(BANDS, city_dir=CITY_DIR)     # CITY_DIR=None -> committed OBJ grid
cs = float(man['cell_size_m']); NX, NY, NZ = grid.shape
corners = [VC.vox_to_lonlat(man, vx, vz) for vx in (0, NX) for vz in (0, NZ)]
lons = [c[0] for c in corners]; lats = [c[1] for c in corners]
sb = defaultdict(set)
for s in sts:
    sb[s['site']].add(s['band'])
n_fields = len(fbs._dedupe_by_site(sts))

print("CRS / georeference (every lon/lat <-> voxel uses this):")
print(f"  EPSG:{man['epsg']} Web-Mercator · anchor = FCC HQ {CG.FCC_LONLAT}")
print(f"  merc_anchor = {[round(a,1) for a in man['merc_anchor']]}  merc_scale = {man.get('merc_scale')}")
print(f"  cell = {cs} m · grid {NX}x{NY}x{NZ} vox · {NX*cs:.0f} x {NZ*cs:.0f} m · "
      f"buildings = {'real OSM' if CITY_DIR else 'OBJ'}")
print(f"  bbox lon/lat  [{min(lons):.5f}, {min(lats):.5f}]  ..  [{max(lons):.5f}, {max(lats):.5f}]")
print(f"\nDataset = {len(sb)} stations x {n_fields} (site,band) fields x {len(ANTENNA_KINDS)} antenna "
      f"waveforms = {n_fields*len(ANTENNA_KINDS)} shards")
for b in BANDS:
    ss = sorted({s['site'].split('_')[0] for s in sts if s['band'] == b})
    print(f"   {b:14} {get(b).f_mhz:6.0f} MHz {get(b).family:4} : {ss or '(no station)'}")
print(f"   antenna waveforms: {ANTENNA_KINDS}")

# y=2 building slice = exactly the plane bs_region crops each Tx region from
plane = np.asarray(grid[:, 2, :]) == 3
fig, ax = plt.subplots(figsize=(12, 12 * NZ / NX))
ax.imshow(plane.T, origin='lower', cmap='Greys', extent=[0, NX*cs, 0, NZ*cs], alpha=.9)
fx, fz = VC.lonlat_to_vox(man, *CG.FCC_LONLAT)
ax.plot(fx*cs, fz*cs, 'k^', ms=13, zorder=8)
done = set()
for s in sts:
    if s['site'] in done:
        continue
    done.add(s['site'])
    fams = {get(bb).family for bb in sb[s['site']]}
    c = 'deepskyblue' if 'WLAN' in fams else ('orange' if s['site'] == VAL_SITE else 'lime')
    ax.plot(s['vx']*cs, s['vz']*cs, '*', color=c, ms=17, mec='k', zorder=6)
    th = np.radians(s['boresight'])                       # boresight arrow (X=west+, Z=north+)
    ax.arrow(s['vx']*cs, s['vz']*cs, -np.sin(th)*90, np.cos(th)*90,
             color='k', head_width=18, length_includes_head=True, zorder=5)
    bl = ' '.join(sorted({bb.split('_')[-1] for bb in sb[s['site']]}))
    ax.text(s['vx']*cs+10, s['vz']*cs+8, f"{s['site'].split('_')[0]} · {bl}", fontsize=7.5, zorder=7)
leg = [Line2D([0],[0], marker='^', color='w', mfc='k', label='FCC HQ (walk Rx)', ms=10),
       Line2D([0],[0], marker='*', color='w', mfc='lime', mec='k', label='cellular Tx (train)', ms=13),
       Line2D([0],[0], marker='*', color='w', mfc='orange', mec='k', label=f'g2 check ({VAL_SITE.split("_")[0]})', ms=13),
       Line2D([0],[0], marker='*', color='w', mfc='deepskyblue', mec='k', label='WLAN AP (artificial)', ms=13)]
ax.legend(handles=leg, loc='upper right')
ax.set_title(f"Grid the dataset & U-Net see · {'OSM' if CITY_DIR else 'OBJ'} buildings (class 3 = barrier) · "
             f"EPSG:{man['epsg']}\n6 stations x 4G/5G/WLAN bands x {len(ANTENNA_KINDS)} antenna waveforms "
             f"(arrow = boresight)")
ax.set_xlabel('X  (m, west +)'); ax.set_ylabel('Z  (m, north +)')
plt.tight_layout(); plt.show()


In [ ]:
# ---- Map + CRS of the grid the Dataset & U-Net see (all 6 stations · 4G/5G/WLAN · antennas) ----
import numpy as np, matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from collections import defaultdict
import _bootstrap as B, voxelize_city as VC, city_georef as CG
import fw_bs_catalog as fbs
from bands_v3 import get

grid, man, sts = fbs.load_stations(BANDS, city_dir=CITY_DIR)     # CITY_DIR=None -> committed OBJ grid
cs = float(man['cell_size_m']); NX, NY, NZ = grid.shape
corners = [VC.vox_to_lonlat(man, vx, vz) for vx in (0, NX) for vz in (0, NZ)]
lons = [c[0] for c in corners]; lats = [c[1] for c in corners]
sb = defaultdict(set)
for s in sts:
    sb[s['site']].add(s['band'])
n_fields = len(fbs._dedupe_by_site(sts))

print("CRS / georeference (every lon/lat <-> voxel uses this):")
print(f"  EPSG:{man['epsg']} Web-Mercator · anchor = FCC HQ {CG.FCC_LONLAT}")
print(f"  merc_anchor = {[round(a,1) for a in man['merc_anchor']]}  merc_scale = {man.get('merc_scale')}")
print(f"  cell = {cs} m · grid {NX}x{NY}x{NZ} vox · {NX*cs:.0f} x {NZ*cs:.0f} m · "
      f"buildings = {'real OSM' if CITY_DIR else 'OBJ'}")
print(f"  bbox lon/lat  [{min(lons):.5f}, {min(lats):.5f}]  ..  [{max(lons):.5f}, {max(lats):.5f}]")
print(f"\nDataset = {len(sb)} stations x {n_fields} (site,band) fields x {len(ANTENNA_KINDS)} antenna "
      f"waveforms = {n_fields*len(ANTENNA_KINDS)} shards")
for b in BANDS:
    ss = sorted({s['site'].split('_')[0] for s in sts if s['band'] == b})
    print(f"   {b:14} {get(b).f_mhz:6.0f} MHz {get(b).family:4} : {ss or '(no station)'}")
print(f"   antenna waveforms: {ANTENNA_KINDS}")

# y=2 building slice = exactly the plane bs_region crops each Tx region from
plane = np.asarray(grid[:, 2, :]) == 3
fig, ax = plt.subplots(figsize=(12, 12 * NZ / NX))
ax.imshow(plane.T, origin='lower', cmap='Greys', extent=[0, NX*cs, 0, NZ*cs], alpha=.9)
fx, fz = VC.lonlat_to_vox(man, *CG.FCC_LONLAT)
ax.plot(fx*cs, fz*cs, 'k^', ms=13, zorder=8)
done = set()
for s in sts:
    if s['site'] in done:
        continue
    done.add(s['site'])
    fams = {get(bb).family for bb in sb[s['site']]}
    c = 'deepskyblue' if 'WLAN' in fams else ('orange' if s['site'] == VAL_SITE else 'lime')
    ax.plot(s['vx']*cs, s['vz']*cs, '*', color=c, ms=17, mec='k', zorder=6)
    th = np.radians(s['boresight'])                       # boresight arrow (X=west+, Z=north+)
    ax.arrow(s['vx']*cs, s['vz']*cs, -np.sin(th)*90, np.cos(th)*90,
             color='k', head_width=18, length_includes_head=True, zorder=5)
    bl = ' '.join(sorted({bb.split('_')[-1] for bb in sb[s['site']]}))
    ax.text(s['vx']*cs+10, s['vz']*cs+8, f"{s['site'].split('_')[0]} · {bl}", fontsize=7.5, zorder=7)
leg = [Line2D([0],[0], marker='^', color='w', mfc='k', label='FCC HQ (walk Rx)', ms=10),
       Line2D([0],[0], marker='*', color='w', mfc='lime', mec='k', label='cellular Tx (train)', ms=13),
       Line2D([0],[0], marker='*', color='w', mfc='orange', mec='k', label=f'g2 check ({VAL_SITE.split("_")[0]})', ms=13),
       Line2D([0],[0], marker='*', color='w', mfc='deepskyblue', mec='k', label='WLAN AP (artificial)', ms=13)]
ax.legend(handles=leg, loc='upper right')
ax.set_title(f"Grid the dataset & U-Net see · {'OSM' if CITY_DIR else 'OBJ'} buildings (class 3 = barrier) · "
             f"EPSG:{man['epsg']}\n6 stations x 4G/5G/WLAN bands x {len(ANTENNA_KINDS)} antenna waveforms "
             f"(arrow = boresight)")
ax.invert_xaxis()   # grid X is west-positive; flip so the map is north-up / east-right like OSM
ax.set_xlabel('metres — map north-up (west ← | → east)'); ax.set_ylabel('metres — north ↑')
plt.tight_layout(); plt.show()


### 5 · GPU acceleration (CuPy) — accelerates the base-station FDTD too
Same cell as indoor: it patches `fw_dataset._run_field`, which `fw_bs_catalog.bs_field` calls, so outdoor generation runs on the GPU unchanged.

In [ ]:
# --- GPU FDTD (CuPy): reuse FullWaveScene setup, run only the time loop on GPU ---
import numpy as np, math
import fw_dataset
from fullwave2d import FullWaveScene

C0 = 299_792_458.0
_cpu_run_field = fw_dataset._run_field          # keep original (fallback + parity check)

USE_GPU = False
try:
    import cupy as cp
    USE_GPU = cp.cuda.runtime.getDeviceCount() > 0
except Exception:
    try:  # Colab GPU runtime usually ships CuPy; install the CUDA-12 wheel if not
        import subprocess, sys
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'cupy-cuda12x'], check=True)
        import cupy as cp
        USE_GPU = cp.cuda.runtime.getDeviceCount() > 0
    except Exception as e:
        print('CuPy unavailable -> staying on CPU:', e)

def _laplacian_gpu(u, inv_h2):                  # matches Spatial_Physics.laplacian (np.roll)
    lap = cp.zeros_like(u)
    for ax in range(u.ndim):
        lap += cp.roll(u, 1, axis=ax) + cp.roll(u, -1, axis=ax)
    lap -= 2.0 * u.ndim * u
    return lap * inv_h2

def _run_field_gpu(classes, h, tx_ij, f_mhz, crossings):
    sim = FullWaveScene(classes, h, f_mhz, tx_ij, source='cw')     # identical CPU setup
    steps = int(round(crossings * max(classes.shape) * h / C0 / sim.dt))
    dt, f0 = sim.dt, sim.f0
    omega = 2.0 * np.pi * f0
    inv_h2 = float(sim.inv_h2)
    u      = cp.asarray(sim.u)                  # move only the fields the loop touches
    u_prev = cp.asarray(sim.u_prev)
    cdt2   = cp.asarray(sim.cdt2)
    inv1pa = cp.asarray(sim._inv1pa)
    _1ma   = cp.asarray(sim._1ma)
    damp   = cp.asarray(sim.damp)
    rigid  = cp.asarray(sim.rigid)
    src    = sim.src_idx
    warmup = int(0.6 * steps)                   # same as _run_field's simulate() call
    period_steps = max(1, int(round((1.0 / f0) / dt)))
    win_start = max(warmup, steps - 2 * period_steps)             # phasor_periods = 2
    acc = cp.zeros(u.shape, cp.complex128); n_win = 0
    for k in range(steps):                      # mirrors FullWaveScene.step() exactly
        lap = _laplacian_gpu(u, inv_h2)
        u_next = (2.0 * u - _1ma * u_prev + cdt2 * lap) * inv1pa
        u_next[src] += math.sin(omega * (k * dt))                # cw() soft source, t=k*dt
        u_next[rigid] = 0.0                     # perfect reflectors
        u_next *= damp                          # absorbing sponge
        u_prev = u * damp
        u = u_next
        if k >= win_start:                      # on-the-fly single-freq DFT (u at (k+1)dt)
            acc += u * complex(np.exp(-1j * omega * (k + 1) * dt))
            n_win += 1
    if not bool(cp.isfinite(u).all()):
        raise FloatingPointError('field blew up (GPU)')
    return cp.asnumpy((2.0 / max(n_win, 1)) * acc)               # complex phasor U, on CPU

if USE_GPU:
    fw_dataset._run_field = _run_field_gpu       # generate() -> _indoor_field -> this
    name = cp.cuda.runtime.getDeviceProperties(0)['name'].decode()
    # parity vs CPU on a tiny field so you can trust the port
    rng = np.random.default_rng(0)
    test = ((rng.random((96, 96)) < 0.12).astype(np.int8) * 2)   # sparse concrete
    Ucpu = _cpu_run_field(test, 0.06, (48, 48), 617.0, 1.2)
    Ugpu = _run_field_gpu(test, 0.06, (48, 48), 617.0, 1.2)
    rel = float(np.abs(Ugpu - Ucpu).max() / (np.abs(Ucpu).max() + 1e-30))
    print(f'GPU FDTD ON -> {name} | parity max|dU|/|U| = {rel:.2e} (want < 1e-6)')
else:
    print('GPU FDTD OFF -> CPU _run_field (pick a GPU runtime for the speed-up).')


### 6 · Phase B — generate from real base stations (directional FDTD)

In [ ]:
import fw_bs_catalog as fbs
fbs.generate_bs(BANDS, holdout_site=HOLDOUT, boxes_per_station=BOXES_PER_STATION,
                out_dir=OUT, max_cells=MAX_CELLS, seed=1, city_dir=CITY_DIR,
                antenna_kinds=ANTENNA_KINDS)   # one omni FDTD per (site,band) x each antenna pattern


### 7 · Phase C — train the U-Net surrogate (AMP, auto 10-channel)

In [ ]:
import glob, fw_unet2d
print(len(glob.glob(OUT + '/shard_*.npz')), 'shards at', OUT)
model, best = fw_unet2d.train(OUT, epochs=EPOCHS, base=BASE, bs=BATCH, out=CKPT)
print('best val_mse =', best)

### 8 · Phase D — g2 on the held-out real station (Forte Hall) + ONNX export

In [ ]:
!pip -q install onnx onnxruntime
import fw_export
model = fw_unet2d.load_model(CKPT)
print(fbs.validate_bs(model, BANDS, holdout_site=VAL_SITE, max_cells=MAX_CELLS))
fw_export.export_onnx(model.cpu(), fw_export.WEB / 'fw_bs.onnx'); print('exported fw_bs.onnx')

### 9 · Directional surrogate coverage per band (no FDTD)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
grid, man, sts = fbs.load_stations(BANDS, city_dir=CITY_DIR)
dev = fw_unet2d.pick_device(); model = model.to(dev)
by_band = {}
for s in fbs._dedupe_by_site(sts): by_band.setdefault(s['band'], s)   # one station per band
bands = list(by_band)
fig, axes = plt.subplots(1, len(bands), figsize=(4.5*len(bands), 4), squeeze=False)
for ax, band in zip(axes.flat, bands):
    st = by_band[band]
    p = fbs.bs_region(grid, man, st, npw=8.0, max_cells=MAX_CELLS)         # region only, no FDTD
    U = fbs._tiled_predict_bs(model, p.classes, p.tx_idx, p.h_m, st['f_mhz'], st['boresight'], st.get('kind','panel'))
    env = 20*np.log10(np.abs(U)+1e-12); env -= env.max()
    im = ax.imshow(env.T, origin='lower', cmap='viridis', vmin=-45, vmax=0,
                   extent=[0, p.extent_m[0], 0, p.extent_m[1]])
    ax.plot(p.tx_idx[0]*p.h_m, p.tx_idx[1]*p.h_m, 'r*', ms=10)
    ax.set_title(f"{st['site']} · {band} · brg {st['boresight']:.0f}°")
    fig.colorbar(im, ax=ax, shrink=0.7); ax.invert_xaxis()
fig.suptitle('Outdoor surrogate coverage (directional, no FDTD) — |U| dB'); fig.tight_layout(); plt.show()

### 11 · RSRP cross-check — surrogate vs walk test (per PCI, per station)
Surrogate path-loss RSRP at the FCC-HQ receiver (FSPL + surrogate building-excess, one fitted EIRP anchor) vs the catalog's measured walk-test RSRP, a few PCIs at each base station. Also `predict_rsrp(site, lon, lat)` for RSRP at any coordinate. Needs the trained `model` (run Phase C/D first).

In [ ]:
# ---- RSRP cross-check: surrogate path-loss RSRP vs walk-test, per PCI per station ----
# Rx = the walk-test reference point (FCC HQ; catalog distance_m is station->FCC HQ).
# Anchor-consistent link budget across stations:
#     RSRP_pred = K  -  [ FSPL_3D(d, f_real)  +  Excess_surrogate(Rx) ]
#   Excess = |U_freespace| - |U_buildings| at the Rx, each from ONE surrogate box
#            (Tx far outside the box = an in-distribution far crop; the per-field
#            99th-pct normalization ~cancels in the air/real ratio).
#   K (= EIRP - cables, one lumped constant) is fit once to all measured points, so
#   the residual measures the surrogate's RELATIVE accuracy, not the absolute anchor.
import csv, json, numpy as np, torch, matplotlib.pyplot as plt
from scipy import ndimage
from scipy.stats import spearmanr
import _bootstrap as B, dataset_3d as D3, city_georef as CG, voxelize_city as VC
from bands_v3 import get

norm = D3.load_norm(json.loads(B.MANIFEST.read_text()))
dev = fw_unet2d.pick_device(); model = model.to(dev).eval()
grid, man, sts = fbs.load_stations(BANDS, city_dir=CITY_DIR)
cs = float(man['cell_size_m']); NX, NZ = grid.shape[0], grid.shape[2]
st_by = {(s['site'], s['band']): s for s in fbs._dedupe_by_site(sts)}
rx_vx, rx_vz = VC.lonlat_to_vox(man, *CG.FCC_LONLAT)          # the receiver point


def surrogate_level(st, rvx, rvz, air, box=128, npw=8.0):
    """20*log10|U| (relative dB) the surrogate predicts at (rvx,rvz), from a single
    box around the Rx. air=True zeroes the buildings -> free-space reference."""
    band = get(st['band']); h = band.cell_size_m(npw); zoom = cs / h
    gw = int(np.ceil(box / zoom)) + 2
    xi = max(0, int(round(rvx - gw / 2))); zi = max(0, int(round(rvz - gw / 2)))
    sub = np.asarray(grid[xi:xi + gw, 2, zi:zi + gw])
    if min(sub.shape) < 2:
        return None
    fine = ndimage.zoom(sub, zoom, order=0).astype(np.int8)
    if fine.shape[0] < box or fine.shape[1] < box:
        fine = np.pad(fine, ((0, max(0, box - fine.shape[0])), (0, max(0, box - fine.shape[1]))))
    fine = fine[:box, :box]
    if air:
        fine = np.zeros_like(fine)
    tx = ((st['vx'] - xi) * zoom, (st['vz'] - zi) * zoom)     # Tx in fine coords (far outside box)
    ii, jj = np.mgrid[0:box, 0:box]
    d_m = np.hypot(ii - tx[0], jj - tx[1]) * h
    x = fbs.featurize_bs(fine, tx, d_m, norm.freq_feature(band.f_mhz), st['boresight'], st.get('kind', 'panel'))
    with torch.no_grad():
        pr = model(torch.from_numpy(x[None]).to(dev)).cpu().numpy()[0]
    c = box // 2
    return 20 * np.log10(np.hypot(pr[0, c, c], pr[1, c, c]) + 1e-12)


def fspl_db(d_m, f_mhz):                                       # 3-D free-space path loss
    return 20 * np.log10(max(d_m, 1.0)) + 20 * np.log10(f_mhz) - 27.55


# building excess loss at the Rx per (site, band) — one air + one real box each
excess = {}
for key, st in st_by.items():
    try:
        Lr, La = surrogate_level(st, rx_vx, rx_vz, False), surrogate_level(st, rx_vx, rx_vz, True)
        if Lr is not None and La is not None:
            excess[key] = max(0.0, La - Lr)
    except Exception as e:
        print('  skip', key, e)

# measured walk-test PCIs that map to a trained SIM band + a predictable station
rows = []
for r in csv.DictReader(open(CG.CATALOG)):
    if not r['rsrp_mean'].strip() or not r['freq_mhz'].strip():
        continue
    lab = (fbs.BAND_TO_LABEL.get(r['freq_mhz'].rstrip('0').rstrip('.'))
           or fbs.BAND_TO_LABEL.get(str(int(float(r['freq_mhz'])))))
    key = (r['site_id'], lab)
    if lab not in BANDS or key not in excess:
        continue
    d, f = float(r['distance_m']), float(r['freq_mhz'])
    rows.append(dict(site=r['site_id'], band=lab, pci=r['pci'], d=d, f=f,
                     rsrp=float(r['rsrp_mean']), pl=fspl_db(d, f) + excess[key]))

if not rows:
    print('No measured PCIs map to the trained bands at a predictable station.')
else:
    pl = np.array([x['pl'] for x in rows]); meas = np.array([x['rsrp'] for x in rows])
    K = float(np.median(meas + pl))                            # lumped EIRP-cables anchor
    for x in rows:
        x['pred'] = K - x['pl']; x['res'] = x['pred'] - x['rsrp']
    res = np.array([x['res'] for x in rows])
    print(f"fitted anchor K (EIRP - cables) = {K:.1f} dBm   [RSRP_pred = K - FSPL - surrogate_excess]\n")
    print(f"{'site':20}{'band':>12}{'pci':>5}{'d_m':>6}{'excdB':>7}{'meas':>8}{'pred':>8}{'resid':>7}")
    for x in sorted(rows, key=lambda r: (r['site'], r['band'])):
        print(f"{x['site']:20}{x['band']:>12}{x['pci']:>5}{x['d']:>6.0f}"
              f"{excess[(x['site'], x['band'])]:>7.1f}{x['rsrp']:>8.1f}{x['pred']:>8.1f}{x['res']:>+7.1f}")
    rho = spearmanr(-pl, meas)[0]
    print(f"\nresidual: MAE {np.abs(res).mean():.1f} dB · RMSE {np.sqrt((res**2).mean()):.1f} dB · "
          f"Spearman(pred, meas) {rho:+.2f} · n={len(rows)} over {len(set(x['site'] for x in rows))} stations")
    print("caveats: the surrogate predicts one field per (site,band), so the RSRP spread between "
          "co-sited PCIs (same sector) is an irreducible noise floor; measured freq is mapped to the "
          "nearest SIM band; Rx = the FCC point vs a walk MEAN; 2-D floor-plane spreading. Read this as "
          "a link-budget sanity check, not a calibrated RSRP model.")
    plt.figure(figsize=(5, 5))
    plt.scatter([x['pred'] for x in rows], meas, c='tab:blue', s=40)
    lo, hi = min(meas.min(), (K - pl).min()) - 3, max(meas.max(), (K - pl).max()) + 3
    plt.plot([lo, hi], [lo, hi], 'k--', lw=1)
    plt.xlabel('surrogate RSRP_pred (dBm)'); plt.ylabel('walk-test RSRP (dBm)')
    plt.title('Surrogate vs walk-test RSRP'); plt.tight_layout(); plt.show()

# evaluate RSRP at ANY lon/lat from a given station (the 'RSRP on a specific coordinate' ask)
K_ANCHOR = K if rows else 62.0
def predict_rsrp(site, lon, lat, band=None, eirp_dbm=None):
    cand = [s for s in fbs._dedupe_by_site(sts) if s['site'] == site and (band is None or s['band'] == band)]
    if not cand:
        return None
    st = cand[0]; vx, vz = VC.lonlat_to_vox(man, lon, lat)
    Lr, La = surrogate_level(st, vx, vz, False), surrogate_level(st, vx, vz, True)
    exc = max(0.0, La - Lr) if (Lr is not None and La is not None) else 0.0
    d = float(np.hypot(vx - st['vx'], vz - st['vz']) * cs)
    return (K_ANCHOR if eirp_dbm is None else eirp_dbm) - fspl_db(d, st['f_mhz']) - exc

print('\nexample · predict_rsrp("BS6_1005_n_capitol", *FCC) =',
      round(predict_rsrp('BS6_1005_n_capitol', *CG.FCC_LONLAT) or float('nan'), 1), 'dBm')


### 10 · Save to Drive — flush + verify

In [ ]:
import shutil, glob
from google.colab import drive
shutil.copytree(OUT, '/content/drive/MyDrive/fw_data_bs', dirs_exist_ok=True)
shutil.copy(CKPT, '/content/drive/MyDrive/fw_bs.pt')
drive.flush_and_unmount(); drive.mount('/content/drive')
print(len(glob.glob('/content/drive/MyDrive/fw_data_bs/shard_*.npz')), 'shards on Drive (verified)')